# 05 — Observability, testing, and performance tuning (EMR)

Operational practices a production on-call engineer actually uses: streaming query metrics, table history/detail, query plans, and data quality checks.

Run the cell below first, before anything else -- it configures Delta Lake for this notebook's Spark session (EMR doesn't bundle Delta by default, unlike Databricks). `%%configure -f` must run before any other Spark code in this session.

In [ ]:
%%configure -f
{"conf": {"spark.jars.packages": "io.delta:delta-spark_2.12:3.1.0", "spark.sql.extensions": "io.delta.sql.DeltaSparkSessionExtension", "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"}}

In [ ]:
schema = "retail_lakehouse"
base_path = "s3://<your-lakehouse-bucket>/data"

from retail_lakehouse.config import PipelineConfig
cfg = PipelineConfig(schema=schema, base_path=base_path)
spark.sql(f"USE `{cfg.schema}`")

## Streaming query metrics

Every `StreamingQuery` exposes `.lastProgress`/`.recentProgress` with metrics you should alert on:

- `numInputRows`/`inputRowsPerSecond` -- throughput; a sudden drop can mean an upstream (MSK) problem.
- `durationMs` per phase (`addBatch`, `getBatch`, `queryPlanning`) -- where time is actually spent.
- `sources[].endOffset` vs the topic's actual latest offset -- this delta is **consumer lag**.
- `stateOperators[].numRowsTotal` -- state store size; unbounded growth usually means a missing/too-long watermark or a `dropDuplicates` key that never converges.

In production, ship these via a `StreamingQueryListener` to CloudWatch/Datadog/Prometheus rather than reading them interactively:

```python
from pyspark.sql.streaming import StreamingQueryListener

class MetricsListener(StreamingQueryListener):
    def onQueryProgress(self, event):
        p = event.progress
        # push p.numInputRows, p.inputRowsPerSecond, p.durationMs.get("addBatch") to your metrics sink
        pass
    def onQueryStarted(self, event): pass
    def onQueryTerminated(self, event): pass

spark.streams.addListener(MetricsListener())
```

## Table history and storage detail

In [ ]:
for tbl in ["bronze_clickstream_kafka", "silver_clickstream_streaming", "gold_revenue_windows_streaming"]:
    try:
        print("==", tbl, "history ==")
        spark.sql(f"DESCRIBE HISTORY {cfg.table(tbl)}").show()
        print("==", tbl, "detail ==")
        spark.sql(f"DESCRIBE DETAIL {cfg.table(tbl)}").show()
    except Exception as e:
        print(tbl, "not available yet:", str(e)[:200])

## Query plan and tuning levers

Look for scan filters (`PushedFilters`), broadcast exchanges, sort-merge joins, shuffle exchanges, and skew hints in `explain("formatted")` output.

In [ ]:
if spark.catalog.tableExists(f"{cfg.schema}.silver_clickstream_batch"):
    q = spark.table(cfg.table("silver_clickstream_batch")).groupBy("event_type").count()
    q.explain("formatted")
    q.show(truncate=False)

## Data quality summary

In [ ]:
from retail_lakehouse.quality import quality_summary, assert_no_duplicate_keys

if spark.catalog.tableExists(f"{cfg.schema}.silver_clickstream_batch"):
    quality_summary(spark.table(cfg.table("silver_clickstream_batch"))).show()
    assert_no_duplicate_keys(spark.table(cfg.table("silver_clickstream_batch")), ["event_id"])
    print("No duplicate event_id in silver_clickstream_batch")

## Production tuning and operations checklist

- Start with a correct data model and partition strategy; fix the model before tuning code.
- Keep file sizes healthy (target ~128MB-1GB per file); `OPTIMIZE` after ingestion bursts, not every micro-batch.
- Use broadcast joins only for genuinely small dimensions; check `explain()` to confirm the plan Spark actually picked, don't assume.
- Trust AQE for partition coalescing and skew joins; only hand-tune `spark.sql.shuffle.partitions` when AQE's defaults are measurably wrong for your workload.
- Cache only reused intermediate data, and `unpersist()` when done.
- Size streaming state and watermarks deliberately -- measure actual event lateness in your data before picking a watermark delay.
- Alert on consumer lag and state store size growth, not just job failure/success.
- Never delete a production streaming checkpoint casually -- it is the source of truth for "what has been processed."

## Next

`06_capstone_end_to_end.ipynb` runs the full pipeline as a single validated flow -- this is also what the MWAA DAG (`airflow/dags/retail_lakehouse_pipeline.py`) orchestrates as a scheduled run.